# 🌳 Segmentação de áreas verdes de Niterói/RJ com **samgeo** (LangSAM)

Demonstração de segmentação por **prompt de texto** usando `segment-geospatial` (GroundingDINO + Segment Anything Model).

**Antes de rodar:** `Ambiente de execução` → `Alterar o tipo de ambiente de execução` → **GPU** (T4 basta).

Fluxo: baixar imagem de satélite da AOI → rodar o modelo com um prompt → visualizar → exportar (raster + vetor).

## 1. Instalação

Dois cuidados de compatibilidade (jan/2026):
- `groundingdino-py` **não** vem como dependência do `segment-geospatial` — instalamos junto.
- `transformers 5.x` removeu o método `get_head_mask` que o GroundingDINO ainda usa, então fixamos o `transformers` em 4.x.

Rode esta célula **antes** de qualquer import. Numa execução limpa não é preciso reiniciar; se você já tinha importado o `transformers` nesta sessão, reinicie o ambiente após instalar.

In [ ]:
# ~2-3 min na primeira execução
%pip install -q segment-geospatial groundingdino-py "transformers==4.57.6" leafmap localtileserver

## 2. Imports e checagem de GPU

In [ ]:
import leafmap
import torch
from samgeo.common import tms_to_geotiff, raster_to_vector  # nas versões novas ficam em samgeo.common
from samgeo.text_sam import LangSAM

print('GPU disponível:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('⚠️ Sem GPU. Ative em Ambiente de execução → Alterar tipo → GPU (o modelo roda na CPU, mas fica lento).')

## 3. Área de interesse (AOI)

Rode a célula abaixo para abrir o mapa interativo e **desenhar um retângulo** sobre a área desejada (use as ferramentas de desenho à esquerda). Se preferir, pule esta célula e use a AOI padrão da célula seguinte.

In [ ]:
m = leafmap.Map(center=[-22.9018, -43.1055], zoom=16, height='600px')
m.add_basemap('SATELLITE')
m

## 4. Definir a bounding box

Se você desenhou um retângulo acima, ele é usado automaticamente. Caso contrário, cai na AOI padrão (Campo de São Bento e entorno, Icaraí — bom contraste verde × urbano).

In [ ]:
# AOI padrão: [oeste, sul, leste, norte]
bbox_padrao = [-43.1105, -22.9060, -43.1010, -22.8975]

if m.user_roi_bounds() is not None:
    bbox = m.user_roi_bounds()   # usa o que foi desenhado no mapa
    print('Usando AOI desenhada no mapa:')
else:
    bbox = bbox_padrao
    print('Usando AOI padrão (Campo de São Bento / Icaraí):')
print(bbox)

## 5. Baixar a imagem de satélite

`zoom=18` dá boa resolução para a demo. Aumente para 19 se a AOI for pequena; diminua para 17 se for grande (mais zoom = mais detalhe, porém mais memória).

In [ ]:
imagem = 'niteroi_satelite.tif'
tms_to_geotiff(output=imagem, bbox=bbox, zoom=18, source='Satellite', overwrite=True)

# Visualiza a imagem baixada
m2 = leafmap.Map()
m2.add_raster(imagem, layer_name='Imagem de satélite')
m2

## 6. Segmentação por prompt de texto

A primeira chamada `LangSAM()` baixa os pesos do GroundingDINO + SAM (~1 min).

**Dicas de prompt:** substantivos concretos funcionam melhor no GroundingDINO. Experimente `"tree"`, `"vegetation"`, `"green vegetation"`, `"forest"`.

- `box_threshold`: quão confiante o detector precisa estar (↑ = menos detecções, mais precisas).
- `text_threshold`: força da associação com o texto (↑ = mais restritivo).

In [ ]:
sam = LangSAM()   # baixa os pesos na primeira vez

In [ ]:
prompt = 'green vegetation'
mascara = 'areas_verdes.tif'

sam.predict(
    image=imagem,
    text_prompt=prompt,
    box_threshold=0.24,
    text_threshold=0.24,
    output=mascara,      # salva a máscara georreferenciada (GeoTIFF)
    mask_multiplier=255,
    dtype='uint8',
)
print('Máscara salva em', mascara)

## 7. Visualizar o resultado

In [ ]:
# Figura estática com as áreas segmentadas sobre a imagem
sam.show_anns(
    cmap='Greens',
    box_color='red',
    title=f'Áreas verdes de Niterói/RJ — prompt: "{prompt}"',
    blend=True,
    output='areas_verdes_anns.png',
)

In [ ]:
# Sobreposição interativa (raster da máscara sobre o satélite)
m3 = leafmap.Map()
m3.add_basemap('SATELLITE')
m3.add_raster(mascara, palette='Greens', opacity=0.6, nodata=0, layer_name='Áreas verdes')
m3

## 8. Exportar como vetor e baixar

Converte a máscara raster em polígonos (GeoPackage), pronto para abrir no QGIS.

In [ ]:
vetor = 'areas_verdes.gpkg'
raster_to_vector(mascara, vetor)
print('Vetor salvo em', vetor)

In [ ]:
# Baixar os arquivos gerados
from google.colab import files
for f in ['areas_verdes.tif', 'areas_verdes.gpkg', 'areas_verdes_anns.png']:
    files.download(f)

---
### Ideias para a palestra
- Troque o `prompt` ao vivo (`"tree"`, `"building"`, `"swimming pool"`, `"road"`) para mostrar o mesmo pipeline detectando coisas diferentes.
- Desenhe a AOI no mapa na hora, sobre um bairro que a plateia conheça.
- Mostre o `.gpkg` aberto no QGIS para conectar com o fluxo GIS tradicional.